In [ ]:
import os
from threading import Thread  # for running the denoiser in parallel
import queue  # 队列
import time
import numpy as np
import torch
import torch.optim
from models.skip import skip  # our network
from utils.utils import *  # auxiliary functions
from utils.data import Data  # class that holds img, psnr, time
from skimage.restoration import denoise_nl_means
from dncnn_models.network_dncnn import DnCNN as net # dncnn net
from dncnn_models.network_ffdnet import FFDNet as net_ffdnet    # FFDNet

import warnings
warnings.filterwarnings("ignore")

# got GPU? - if you are not getting the exact article results set CUDNN to False
CUDA_FLAG = True
CUDNN = True
if CUDA_FLAG:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    # GPU accelerated functionality for common operations in deep neural nets
    torch.backends.cudnn.enabled = CUDNN
    torch.backends.cudnn.benchmark = CUDNN
    # torch.backends.cudnn.deterministic = True
    dtype = torch.cuda.FloatTensor
else:
    dtype = torch.FloatTensor


ORIGINAL = 'Clean'
CORRUPTED = 'Noisy'
FFDnet= 'FFDnet'
DIP_FFDNET= 'DIP_FFDNET'

def get_network_and_input(img_shape, input_depth=32, pad='reflection',
                          upsample_mode='bilinear', use_interpolate=True, align_corners=False,
                          act_fun='LeakyReLU', skip_n33d=128, skip_n33u=128, skip_n11=4,
                          num_scales=5, downsample_mode='stride', INPUT='noise'):  # 'meshgrid'
    """ Getting the relevant network and network input (based on the image shape and input depth)
        We are using the same default params as in DIP article
        img_shape - the image shape (ch, x, y)
    """
    n_channels = img_shape[0]
    net = skip(input_depth, n_channels,
               num_channels_down=[skip_n33d] * num_scales if isinstance(skip_n33d, int) else skip_n33d,
               num_channels_up=[skip_n33u] * num_scales if isinstance(skip_n33u, int) else skip_n33u,
               num_channels_skip=[skip_n11] * num_scales if isinstance(skip_n11, int) else skip_n11,
               upsample_mode=upsample_mode, use_interpolate=use_interpolate, align_corners=align_corners,
               downsample_mode=downsample_mode, need_sigmoid=True, need_bias=True, pad=pad, act_fun=act_fun).type(dtype)
    net_input = get_noise(input_depth, INPUT, img_shape[1:]).type(dtype).detach()
    return net, net_input


def FFDNet_color_yuan(noisy_np_img,sigma):
    noisy_torch_img=np_to_torch(noisy_np_img)
    n_channels=noisy_torch_img.shape[1]
    denoised_img=[]
    sigma_map=torch.full((1,1,1,1),sigma/255.).type_as(noisy_torch_img) # size:[1,1,1,1]
    #print(noisy_torch_img.shape)  # size:[1,3,256,256]
    # print(noisy_torch_img[:,c,:,:].shape)                 # tensor, size:[1,256,256]
    #temp=torch.unsqueeze(noisy_torch_img[:,c,:,:],dim=0)  # tensor, size:[1,1,256,256] (增加一维)
    
    denoise_torch_fast=model_ffdnet_color(noisy_torch_img,sigma_map)                # denoise_torch_fast:[1,1,256,256]
    #print(denoise_torch_fast.shape)

    denoise_np_fast=torch_to_np(denoise_torch_fast)       # denoise_np_fast:[1,256,256]
    #print(denoise_np_fast.shape)
    denoised_img+=[denoise_np_fast]   
    return np.array(denoised_img, dtype=np.float32)

def train_via_admm(net, net_input, denoiser_function, y, org_img=None,                      # y is the noisy image
                   algorithm_name="", admm_iter=3000, save_path="",           # path to save params
                   LR=0.008,                                     # learning rate
                   sigma_f=3, update_iter=10, method='fixed_point',   # method: 'fixed_point' or 'grad' or 'mixed'
                   beta=.5, mu=.5, LR_x=None, noise_factor=0.033,        # LR_x needed only if method!=fixed_point
                   threshold=20, threshold_step=0.01, increase_reg=0.03):                # increase regularization 
    
    # get optimizer and loss function:
    mse = torch.nn.MSELoss().type(dtype)  # using MSE loss
    # additional noise added to the input:
    net_input_saved = net_input.detach().clone()
    noise = net_input.detach().clone()

    if org_img is not None:
        psnr_y = compare_psnr(org_img, y)  # get the noisy image psnr
        
    # x update method:
    if method == 'fixed_point':
        swap_iter = admm_iter + 1
        LR_x = None
    elif method == 'grad':
        swap_iter = -1
    elif method == 'mixed':
        swap_iter = admm_iter // 2
    else:
        assert False, "method can be 'fixed_point' or 'grad' or 'mixed' only "
    
    # optimizer and scheduler
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)  # using ADAM opt
    y_torch = np_to_torch(y).type(dtype)
    x = y.copy()
    u = np.zeros_like(y)
    f_x = x.copy()
    avg = np.rint(y)

    psnr_net_list=[]
    psnr_x_list  =[]
    psnr_x_u_list=[]
    psnr_avg_list=[]
    image_list  = []

    for i in range(1, 1 + admm_iter):
        # step 1, update network:
        optimizer.zero_grad()
        net_input = net_input_saved + (noise.normal_() * noise_factor)
        
        out = net(net_input)         #原始DIP结果
        out_np = torch_to_np(out)    #转化numpy为了计算psnr
        # loss:
        loss_y = mse(out, y_torch)
        loss_x = mse(out, np_to_torch(x - u).type(dtype))  # -的效果好于+
        total_loss = loss_y + mu * loss_x                  # 新的Loss
        total_loss.backward()
        optimizer.step()


        # step 2, update x using a denoiser and result from step 1
        f_x = denoiser_function(x.copy(), sigma_f)
        
        # 使用深度先验的话需要去掉一维：
        f_x=np.squeeze(f_x)

        if i < swap_iter:
            x = 1 / (beta + mu) * (beta * f_x + mu * (out_np + u))
        else:
            x = x - LR_x * (beta * (x - f_x) + mu * (x - out_np - u))

        np.clip(x, 0, 1, out=x)  # making sure that image is in bounds

        # step 3, update u
        u = u + out_np - x  

        # Averaging: 等同于DIP
        avg = avg * .99 + out_np * .01

        # show psnrs:
        psnr_noisy = compare_psnr(out_np, y)
        if psnr_noisy > threshold:
            mu = mu + increase_reg
            beta = beta + increase_reg
            threshold += threshold_step

        if org_img is not None:
            psnr_avg = compare_psnr(org_img, avg)
            
            
            psnr_avg_list.append(psnr_avg)
            image_list.append(avg)
            psnr_max_temp=max(psnr_avg_list)                       
            psnr_max_temp_index=psnr_avg_list.index(psnr_max_temp)  
            print('\r', algorithm_name, '%04d/%04d Loss %f' % (i, admm_iter, total_loss.item()),'psnrs: y: %.2f avg: %.2f max_psnr_temp: %.2f iteration_number: %.2f' % (psnr_y, psnr_avg,psnr_max_temp,psnr_max_temp_index), end='')
            
            # print('\r','最佳的PSNR:%s,with iteration number:%s'% (psnr_max_temp,psnr_max_temp_index), end='')

        else:
            print('\r', algorithm_name, 'iteration %04d/%04d Loss %f' % (i, admm_iter, total_loss.item()), end='')
    
    return avg,psnr_avg_list,image_list



def run_and_plot(denoiser, name):
    global data_dict
    net, net_input = get_network_and_input(img_shape=data_dict[CORRUPTED].img.shape)
    denoised_img,psnr_avg_list, image_list = train_via_admm(net, net_input, denoiser, y=data_dict[CORRUPTED].img, 
                                                                                        algorithm_name=name,  #method='fixed_point', # method: 'fixed_point' or 'grad' or 'mixed'
                                                                                        admm_iter=iteration_number, #总的迭代数
                                                                                        LR=0.008, # 0.008
                                                                                        sigma_f=25, #3
                                                                                        update_iter=10, 
                                                                                        method='fixed_point',   # method: 'fixed_point' or 'grad' or 'mixed'
                                                                                        beta=0.1, # 可调
                                                                                        mu=0.1,   # 可调
                                                                                        LR_x=None, noise_factor=0.033,        # LR_x needed only if method!=fixed_point
                                                                                        threshold=50, 
                                                                                        threshold_step=0.01, 
                                                                                        increase_reg=0.03,
                                                                                        org_img=data_dict[ORIGINAL].img)
    
    data_dict[name] = Data(denoised_img, compare_psnr(data_dict[ORIGINAL].img, denoised_img))
    #plot_dict(data_dict)
    return psnr_avg_list,image_list

T1 = time.time()

# 处理rgb图像的FFDNet先验：
n_channels_color = 3  #处理单通道图像
# 实例化：
model_ffdnet_color = net_ffdnet(in_nc=n_channels_color,out_nc=n_channels_color,nc=96,nb=12,act_mode='R')
#预训练的FFDNet模型参数：(使用绝对路径！)
model_path_ffdnet_color='/home/yuanweimin/PHD_3/2019_ICCVW_DeepRED/model_zoo/ffdnet_color.pth'

#加载训练参数：
model_ffdnet_color.load_state_dict(torch.load(model_path_ffdnet_color),strict=True)
model_ffdnet_color.eval()
#通常在实际代码中，在预测阶段，也会加上torch.no_grad()来关闭梯度的计算
for k, v in model_ffdnet_color.named_parameters():
    v.requires_grad = False

# load the image and add noise
SIGMA = 15

def load_image(fclean, fnoisy):
    _, img_np = load_and_crop_image(fclean)
    _, img_noisy_np = load_and_crop_image(fnoisy)
    data_dict = {ORIGINAL: Data(img_np), CORRUPTED: Data(img_noisy_np, compare_psnr(img_np, img_noisy_np))}
    initial_psnr = compare_psnr(img_np, img_noisy_np)
    print('Initial PSNR:%s'% initial_psnr)
    return data_dict

fclean_path = '/home/yuanweimin/PHD_4/CC15/13_mean.png'
fnoisy_path = '/home/yuanweimin/PHD_4/CC15/13_real.png'
data_dict = load_image(fclean_path, fnoisy_path)

# 预计总的迭代次数：
iteration_number = 10000

psnr_avg_list,image_list = run_and_plot(FFDNet_color_yuan, DIP_FFDNET)  # you may try it with different denoisers

T2 = time.time()
print('程序运行一次迭代循环的时间:%s秒' % ((T2 - T1)/iteration_number))

# 获取psnr_avg_list列表的最大值和对应索引
psnr_max=max(psnr_avg_list)                       
psnr_max_index=psnr_avg_list.index(psnr_max)   
print('最佳的PSNR:%s,with iteration number:%s'% (psnr_max,psnr_max_index))


# # 获取最好的图像：
# optimal_img=image_list[psnr_max_index]
# np.clip(optimal_img, 0, 1, out=optimal_img) 
# plt.imsave('./OURS.png',optimal_img.transpose(1,2,0))

# plt.imsave(os.path.join(E_path, img_name+ext),(optimal_img).transpose(1,2,0))

In [ ]:
# # 获取最好的图像：
# optimal_img=image_list[100]
# np.clip(optimal_img, 0, 1, out=optimal_img) 
# plt.imsave('./OURS.png',optimal_img.transpose(1,2,0))